In [63]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
import time
import joblib
import MetaTrader5 as mt5
from bot_functions import *
import csv
import os
from datetime import datetime, timedelta, timezone

# Conectar com MT5

# Display data on the MetaTrader 5 package
print("MetaTrader5 package author: ", mt5.__author__)
print("MetaTrader5 package version: ", mt5.__version__)

# Establish connection to the MetaTrader 5 terminal
# You can specify a path to the terminal executable if it's not in the default location
mt5.initialize(path="C:\Program Files\MetaTrader 5 IC Markets EU\terminal64.exe")
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

# Get basic terminal info
print(mt5.terminal_info())

# Get account info (useful to check if logged in)
account_info = mt5.account_info()
if account_info:
    print("\nAccount Info:")
    print(f"  Login: {account_info.login}")
    print(f"  Balance: {account_info.balance}")
    print(f"  Equity: {account_info.equity}")
    print(f"  Free Margin: {account_info.margin_free}")
else:
    print("Failed to get account info, error code =", mt5.last_error())

# At the end of your script or when done, shut down the connection
# mt5.shutdown()

MetaTrader5 package author:  MetaQuotes Ltd.
MetaTrader5 package version:  5.0.5050
TerminalInfo(community_account=False, community_connection=False, connected=True, dlls_allowed=False, trade_allowed=True, tradeapi_disabled=False, email_enabled=False, ftp_enabled=False, notifications_enabled=False, mqid=True, build=5120, maxbars=100000, codepage=0, ping_last=127010, community_balance=0.0, retransmission=0.1911692155895412, company='IC Markets (EU) Ltd', name='MetaTrader 5 IC Markets EU', language='English', path='C:\\Program Files\\MetaTrader 5 IC Markets EU', data_path='C:\\Users\\marti\\AppData\\Roaming\\MetaQuotes\\Terminal\\4B1CE69F577705455263BD980C39A82C', commondata_path='C:\\Users\\marti\\AppData\\Roaming\\MetaQuotes\\Terminal\\Common')

Account Info:
  Login: 52385541
  Balance: 19998.37
  Equity: 19998.8
  Free Margin: 18332.13


In [6]:
# ir buscar o modelo
loaded_bundle = joblib.load('eurusd_model.joblib')
print("Bundle loaded.")
# Access individual components from the loaded bundle
model = loaded_bundle['model']
sl_tp_map = loaded_bundle['sl_tp_map']
avg_duration_by_class = loaded_bundle['avg_duration_by_class']
scaler = loaded_bundle['scaler']

#put parameters
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M5
mt5.symbol_select(SYMBOL, True)
class_to_direction = {0: -1, 1: -1, 2: 0, 3: 1, 4: 1}
# ver isto...................
DEVIATION = 10
risk_per_trade_percentage = 0.01
threshold = 0.7

# to log the trades
log_file = "trade_log.csv"

# Initialize log file with headers if not exists
if not os.path.exists(log_file):
    with open(log_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "symbol", "direction", "signal", "confidence", "lot_size", "sl", "tp", "profit", "duration_sec", "status"])


Bundle loaded.


In [98]:
def close_trade(avg_duration, open_positions, SYMBOL, DEVIATION):

    max_duration = avg_duration * 5 * 60

    for position in open_positions:

        open_time = datetime.fromtimestamp(position.time, tz=timezone.utc)
        broker_now = datetime.fromtimestamp(mt5.symbol_info_tick(SYMBOL).time, tz=timezone.utc)

        duration_sec = (broker_now - open_time).total_seconds()

        if duration_sec > max_duration:
            current_price = (
                mt5.symbol_info_tick(SYMBOL).bid
                if position.type == mt5.ORDER_TYPE_BUY
                else mt5.symbol_info_tick(SYMBOL).ask
            )

            close_type = (
                mt5.ORDER_TYPE_SELL if position.type == mt5.ORDER_TYPE_BUY
                else mt5.ORDER_TYPE_BUY
            )

            close_order = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": SYMBOL,
                "volume": position.volume,
                "type": close_type,
                "price": current_price,
                "deviation": DEVIATION,
                "magic": 123456,
                "comment": "Auto-close by duration",
                "type_time": mt5.ORDER_TIME_GTC,
                "type_filling": mt5.ORDER_FILLING_IOC,
            }

            mt5.order_send(close_order)

In [ ]:
# run the model
while True:
    open_positions = mt5.positions_get(symbol=SYMBOL)
    
    if open_positions is None or len(open_positions) == 0:
        account_info = mt5.account_info()
        df = get_latest_data(SYMBOL, TIMEFRAME, 50)
        X = build_dataset(df)
        X_scaled = scale(X, scaler)
        X_treat = create_lstm_sequences(X_scaled, 12)
        last_candle = X_treat[-1:].copy()
        # last_candle = np.reshape(last_candle, (1, last_candle.shape[1], last_candle.shape[2])).astype(np.float32)

        prediction = model.predict(last_candle)
        signal = np.argmax(prediction, axis=1).item()
        confidence = np.max(prediction, axis=1).item()
        sltp = sl_tp_map.get(signal, {'sl': None, 'tp': None})
        sl = sltp['sl']
        tp = sltp['tp']
        avg_duration = avg_duration_by_class.get(signal, 0)
        balance = account_info.equity - 10000
        direction = class_to_direction.get(signal, 0)
        print(f"Signal: {signal}, Confidence: {confidence:.2f}")

        if direction == 1 and confidence >= threshold:
            lot_size_multiplier = calculate_lot_size_multiplier(sl, balance, risk_per_trade_percentage)
            if lot_size_multiplier:
                execute_trade(sl, tp, direction, lot_size_multiplier, SYMBOL, DEVIATION)
                entry_time = datetime.now()

                # Log trade entry
                with open(log_file, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        entry_time.isoformat(), SYMBOL, direction, signal, confidence,
                        lot_size_multiplier, sl, tp, "", "", "opened"
                    ])
                print(f"Trade executed at {entry_time}, SL: {sl}, TP: {tp}, Lots: {lot_size_multiplier}")
            else:
                print("Lot size too small, trade skipped.")
                
        else:
            close_trade(avg_duration, open_positions, SYMBOL, DEVIATION)

        time.sleep(300)  # espera 5 minutos

Bundle loaded.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Signal: 4, Confidence: 0.57
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
Signal: 1, Confidence: 0.36
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
Signal: 3, Confidence: 0.39
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Signal: 3, Confidence: 0.44
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Signal: 3, Confidence: 0.43
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Signal: 3, Confidence: 0.36
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Signal: 3, Confidence: 0.34
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Signal: 1, Confidence: 0.31
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Signal: 1, Confidence: 0.33
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Signal: 0, Confidence: 0.36
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Signal: 0, Confidence: 0.46
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Signal: 0, Confidence: 0.68
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Signal: 0, Confidence: 0.71
